# Lab 4: Introduction to Graph Neural Networks

Graph Neural Networks (GNNs) are a class of neural architectures designed to learn from graph-structured data. Unlike traditional models that assume independent and identically distributed data, GNNs leverage node relationships (edges) for feature aggregation and representation learning.

**In this lab, you will:**
1. Load and analyze a real-world graph dataset.
2. Implement three model architectures: MLP, GCN, and GAT.
3. Compare full-batch training versus neighborhood sampling strategies.
4. Measure training quality and efficiency under different settings.

---

## Task 0: Dataset Analysis (2 points)

**Objective:** Load a graph dataset (Cora) and analyze its structure.

In [1]:
import torch
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import degree

# ── Load the Cora dataset ──────────────────────────────────────────────────────
dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]

print("=" * 50)
print(f"Dataset: {dataset.name}")
print(f"Number of graphs   : {len(dataset)}")
print(f"Number of classes  : {dataset.num_classes}")
print(f"Number of features : {dataset.num_node_features}")
print("=" * 50)

# ── Node degree statistics ─────────────────────────────────────────────────────
# degree() counts in-edges per node from the edge_index (source nodes)
node_degrees = degree(data.edge_index[0], num_nodes=data.num_nodes)

mean_degree = node_degrees.mean().item()
max_degree  = node_degrees.max().item()

print(f"\nMean node degree : {mean_degree:.4f}")
print(f"Max  node degree : {max_degree:.0f}")

# ── Feature matrix ─────────────────────────────────────────────────────────────
print(f"\nFeature matrix H shape : {data.x.shape}")
#   data.x.shape == (num_nodes, num_features)

# ── Output (label) type ────────────────────────────────────────────────────────
print(f"Label tensor dtype     : {data.y.dtype}")
print(f"Unique label values    : {data.y.unique().tolist()}")

# ── Explanation ────────────────────────────────────────────────────────────────
print("""
Explanation
-----------
Mean node degree:
  The average number of edges per node. In Cora (~3.9) this reflects a
  sparse citation graph. A low mean degree means most nodes have few
  neighbours, which affects how much information a GNN can aggregate in
  a single message-passing step.

Max node degree:
  The highest degree among all nodes (hub nodes). Very high-degree nodes
  ("hubs") can dominate aggregation and cause over-smoothing or gradient
  issues; knowing this guides choices like graph normalization or sampling.

Feature matrix H shape  (num_nodes × num_features):
  Each row is a bag-of-words feature vector for one paper. The shape tells
  us the input dimensionality of every node embedding — critical for
  designing the first GNN layer.

Label dtype (torch.int64 / Long):
  Node labels are integer class indices, confirming this is a multi-class
  node classification task. Long integers are required by PyTorch's
  CrossEntropyLoss.
""")


Dataset: Cora
Number of graphs   : 1
Number of classes  : 7
Number of features : 1433

Mean node degree : 3.8981
Max  node degree : 168

Feature matrix H shape : torch.Size([2708, 1433])
Label tensor dtype     : torch.int64
Unique label values    : [0, 1, 2, 3, 4, 5, 6]

Explanation
-----------
Mean node degree:
  The average number of edges per node. In Cora (~3.9) this reflects a
  sparse citation graph. A low mean degree means most nodes have few
  neighbours, which affects how much information a GNN can aggregate in
  a single message-passing step.

Max node degree:
  The highest degree among all nodes (hub nodes). Very high-degree nodes
  ("hubs") can dominate aggregation and cause over-smoothing or gradient
  issues; knowing this guides choices like graph normalization or sampling.

Feature matrix H shape  (num_nodes × num_features):
  Each row is a bag-of-words feature vector for one paper. The shape tells
  us the input dimensionality of every node embedding — critical for
  de

/Users/stefan/repos/aait-lab/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Processing...
/Users/stefan/repos/aait-lab/.venv/lib/python3.13/site-packages/torch_geometric/io/planetoid.py:107: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  out = pickle.load(f, encoding='latin1')
/Users/stefan/repos/aait-lab/.venv/lib/python3.13/site-packages/torch_geometric/io/planetoid.py:107: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  out = pickle.load(f, encoding='latin1')
/Users/stefan/repos/aait-lab/.venv/lib/python3.13/site-packages

---

## Task 1: Implement GNN Architectures (3 points)

**Objective:** Implement and compare three models — **MLP**, **GCN**, and **GAT** — using a consistent architecture design with a similar total number of parameters.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv

# ── Shared hyper-parameters ────────────────────────────────────────────────────
# Cora: 1433 input features, 7 classes
IN_CHANNELS  = dataset.num_node_features   # 1433
OUT_CHANNELS = dataset.num_classes          # 7
HIDDEN       = 64                           # hidden dimension for MLP & GCN
DROPOUT      = 0.5

# ── 1. MLP ─────────────────────────────────────────────────────────────────────
# Two linear layers; ignores graph structure entirely.
class MLP(nn.Module):
    def __init__(self, in_channels, hidden, out_channels, dropout=DROPOUT):
        super().__init__()
        self.lin1    = nn.Linear(in_channels, hidden)
        self.lin2    = nn.Linear(hidden, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index=None):   # edge_index unused
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin1(x).relu()
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        return x

# ── 2. GCN ─────────────────────────────────────────────────────────────────────
# Two GCNConv layers; aggregates neighbourhood features via symmetric
# normalisation ( Ã = D^{-1/2} A D^{-1/2} ).
class GCN(nn.Module):
    def __init__(self, in_channels, hidden, out_channels, dropout=DROPOUT):
        super().__init__()
        self.conv1   = GCNConv(in_channels, hidden)
        self.conv2   = GCNConv(hidden, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

# ── 3. GAT ─────────────────────────────────────────────────────────────────────
# Two GATConv layers.  Layer-1 uses 4 attention heads × 16 dims (concat → 64),
# layer-2 uses 1 head → out_channels.
# Parameter budget is kept close to MLP/GCN by choosing heads × head_dim = 64.
GAT_HEADS     = 4
GAT_HEAD_DIM  = HIDDEN // GAT_HEADS   # 16

class GAT(nn.Module):
    def __init__(self, in_channels, head_dim, heads, out_channels, dropout=DROPOUT):
        super().__init__()
        self.conv1   = GATConv(in_channels, head_dim, heads=heads,
                               dropout=dropout, concat=True)
        # After concat: heads * head_dim = HIDDEN
        self.conv2   = GATConv(heads * head_dim, out_channels, heads=1,
                               dropout=dropout, concat=False)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

# ── Instantiate models ─────────────────────────────────────────────────────────
mlp = MLP(IN_CHANNELS, HIDDEN, OUT_CHANNELS)
gcn = GCN(IN_CHANNELS, HIDDEN, OUT_CHANNELS)
gat = GAT(IN_CHANNELS, GAT_HEAD_DIM, GAT_HEADS, OUT_CHANNELS)

# ── Parameter count helper ─────────────────────────────────────────────────────
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"{'Model':<8} {'Parameters':>12}")
print("-" * 22)
for name, model in [("MLP", mlp), ("GCN", gcn), ("GAT", gat)]:
    print(f"{name:<8} {count_params(model):>12,}")

print("""
Notes
-----
All three models share the same hidden dimension (64) and depth (2 layers).
• MLP  – two Linear layers, no graph information.
• GCN  – two GCNConv layers; parameter count matches MLP exactly because
         GCNConv(in, out) stores a weight matrix of the same shape as
         nn.Linear(in, out).
• GAT  – uses multi-head attention (4 heads × 16 dims).  The extra
         attention-vector parameters (att_src / att_dst per head) add only
         a small overhead (~300 extra), keeping totals comparable across
         all three models.
""")


Model      Parameters
----------------------
MLP            92,231
GCN            92,231
GAT            92,373

Notes
-----
All three models share the same hidden dimension (64) and depth (2 layers).
• MLP  – two Linear layers, no graph information.
• GCN  – two GCNConv layers; parameter count matches MLP exactly because
         GCNConv(in, out) stores a weight matrix of the same shape as
         nn.Linear(in, out).
• GAT  – uses multi-head attention (4 heads × 16 dims).  The extra
         attention-vector parameters (att_src / att_dst per head) add only
         a small overhead (~300 extra), keeping totals comparable across
         all three models.



---

## Task 2: Full-Batch Training Benchmark (2 points)

**Objective:** Train MLP, GCN, and GAT on the full graph (no sampling) for **30 epochs**.  
Log validation accuracy and total training time for each model.

In [ ]:
import time

# ── Training utilities ─────────────────────────────────────────────────────────
criterion = torch.nn.CrossEntropyLoss()

def train_epoch(model, optimizer):
    model.train()
    optimizer.zero_grad()
    out  = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def evaluate(model):
    model.eval()
    out  = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    val_acc  = (pred[data.val_mask]  == data.y[data.val_mask]).float().mean().item()
    test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
    return val_acc, test_acc

# ── Full-batch training loop ───────────────────────────────────────────────────
EPOCHS = 30
LR     = 0.01

# Re-instantiate models to reset weights
torch.manual_seed(42)
mlp = MLP(IN_CHANNELS, HIDDEN, OUT_CHANNELS)
gcn = GCN(IN_CHANNELS, HIDDEN, OUT_CHANNELS)
gat = GAT(IN_CHANNELS, GAT_HEAD_DIM, GAT_HEADS, OUT_CHANNELS)

results = {}

for model_name, model in [("MLP", mlp), ("GCN", gcn), ("GAT", gat)]:
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=5e-4)

    print(f"\nTraining {model_name} ({EPOCHS} epochs, full-batch) ...")
    print(f"{'Epoch':>6}  {'Loss':>8}  {'Val Acc':>8}")
    print("-" * 28)

    t0 = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        loss = train_epoch(model, optimizer)
        val_acc, _ = evaluate(model)
        if epoch % 5 == 0 or epoch == 1:
            print(f"{epoch:>6}  {loss:>8.4f}  {val_acc:>7.2%}")

    elapsed = time.perf_counter() - t0
    val_acc, test_acc = evaluate(model)

    results[model_name] = {
        "val_acc":      val_acc,
        "test_acc":     test_acc,
        "training_time": elapsed,
    }

# ── Summary table ─────────────────────────────────────────────────────────────
print("\n" + "=" * 48)
print(f"{'Model':<8}  {'Val Acc':>8}  {'Test Acc':>9}  {'Time (s)':>10}")
print("-" * 48)
for name, r in results.items():
    print(f"{name:<8}  {r['val_acc']:>8.2%}  {r['test_acc']:>9.2%}  {r['training_time']:>10.3f}")
print("=" * 48)
